# B1 태그 정제 확인

공통 `load_book_tags()` 결과에서 내용과 관계없는 책장·매체 태그를 제거합니다.
개인 평가와 독서 연도 제거 여부는 B4에서 비교할 수 있도록 `extended`로 나눕니다.
원본 CSV와 다른 팀원의 코드는 수정하지 않습니다.


In [1]:
from pathlib import Path
import json
import pandas as pd
from bookrec.data import load_book_tags, load_books
from bookrec.data.tag_filter import filter_book_tags, tag_filter_report

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "results" / "content_based"
OUT.mkdir(parents=True, exist_ok=True)
tags = load_book_tags()
books = load_books()
print(tags.head().to_string(index=False))


 book_id          tag_name  count
       1         favorites  50755
       1 currently-reading  35418
       1       young-adult  25968
       1           fiction  13819
       1         dystopian  12985


## 1. 규칙별 제거 사례

빈도만으로 제거하지 않습니다. 장르·주제는 남기고 읽기 상태, 소유, 매체 등을 명시적으로 걸러 냅니다. 일치하지 않은 개인 책장 태그가 남을 수 있습니다.


In [2]:
report = tag_filter_report(tags)
removed = report[report["reason"] != "keep"]
print(removed.head(25).to_string(index=False))
removed.to_csv(OUT / "B1_removed_tags.csv", index=False)


         tag_name  books     count       reason
          to-read   9983 140718761        shelf
        favorites   9881   4503173     personal
            owned   9857   1218214        shelf
      books-i-own   9799   1317235        shelf
currently-reading   9776   7507958        shelf
          library   9415    429171        shelf
      owned-books   9221    417384        shelf
           to-buy   8692    363276        shelf
           kindle   8316    506882       format
          default   8238    405327        shelf
            ebook   8053    302572       format
         my-books   7561    225606        shelf
        audiobook   7242    388991       format
           ebooks   7203    217716       format
        wish-list   7192    175729        shelf
       my-library   7000    168884        shelf
       audiobooks   6862    307154       format
            i-own   6669    184728        shelf
            audio   6548    264746       format
       favourites   6422    423289     p

## 2. 정제 전후 통계

태그가 사라진 책도 B2에서 전체 책 목록에 남겨야 합니다.


In [3]:
rows = []
for name, extended in [("basic", False), ("extended", True)]:
    cleaned = filter_book_tags(tags, extended=extended)
    rows.append({
        "mode": name,
        "rows_before": len(tags), "rows_after": len(cleaned),
        "tags_before": tags["tag_name"].nunique(), "tags_after": cleaned["tag_name"].nunique(),
        "books_without_tags": len(set(books["book_id"]) - set(cleaned["book_id"])),
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
summary.to_csv(OUT / "B1_filter_summary.csv", index=False)


    mode  rows_before  rows_after  tags_before  tags_after  books_without_tags
   basic       999898      783564        34250       33203                   0
extended       999898      678861        34250       33103                   0


## 3. 대표 책의 정제 전후

상위 태그를 나란히 확인하고, 잘못 제거된 내용 태그가 없는지 점검합니다.


In [4]:
cleaned = filter_book_tags(tags)
for book_id in [1, 2, 3, 100, 10000]:
    title = books.loc[books["book_id"] == book_id, "title"].iloc[0]
    before = tags.loc[tags["book_id"] == book_id, "tag_name"].head(12).tolist()
    after = cleaned.loc[cleaned["book_id"] == book_id, "tag_name"].head(12).tolist()
    print(f"\n{book_id}: {title}\nbefore: {before}\nafter: {after}")



1: The Hunger Games (The Hunger Games, #1)
before: ['favorites', 'currently-reading', 'young-adult', 'fiction', 'dystopian', 'to-read', 'dystopia', 'fantasy', 'ya', 'science-fiction', 'books-i-own', 'sci-fi']
after: ['young-adult', 'fiction', 'dystopian', 'dystopia', 'fantasy', 'ya', 'science-fiction', 'sci-fi', 'series', 'romance', 'adventure', 'hunger-games']

2: Harry Potter and the Sorcerer's Stone (Harry Potter, #1)
before: ['to-read', 'favorites', 'fantasy', 'currently-reading', 'young-adult', 'fiction', 'harry-potter', 'books-i-own', 'owned', 'ya', 'series', 'favourites']
after: ['fantasy', 'young-adult', 'fiction', 'harry-potter', 'ya', 'series', 'magic', 'childrens', 'children', 'adventure', 'classics', 'children-s']

3: Twilight (Twilight, #1)
before: ['young-adult', 'fantasy', 'favorites', 'vampires', 'ya', 'fiction', 'to-read', 'paranormal', 'books-i-own', 'vampire', 'twilight', 'paranormal-romance']
after: ['young-adult', 'fantasy', 'vampires', 'ya', 'fiction', 'paranorma